# AfriMeet AI — Whisper Fine-Tuning (Colab GPU)

Hybrid workflow: data download/prep can run locally or here; this notebook does the
GPU-heavy parts — Phase 3 (baseline evaluation), Phase 4 (fine-tuning), and Phase 5
(comparison) — using Colab's free GPU. It doesn't duplicate any pipeline logic: every
cell below just calls into the same `afrimeet` package and `scripts/` used locally.
Google Drive is used to persist data and checkpoints between sessions, since Colab's
local disk is wiped when the runtime disconnects.

**Before you start:**
1. `Runtime -> Change runtime type -> GPU` (T4 is fine).
2. The code is cloned straight from the public GitHub repo (`REPO_URL` below) — no
   authentication needed.
   - If you'd rather not clone from GitHub, clear `REPO_URL` to fall back to the
     Drive-zip method instead: run `python scripts/package_for_colab.py` locally and
     upload `dist/afrimeet-ai-code.zip` to `My Drive/AfriMeet_AI/afrimeet-ai-code.zip`.

Phase 6 (the API / web app) isn't part of this notebook — it's meant to run wherever
you deploy it, using the fine-tuned model this notebook produces.

In [ ]:
# Confirm a GPU is attached to this runtime (Runtime -> Change runtime type -> GPU
# if this errors or shows no devices).
!nvidia-smi

In [ ]:
# Mount Google Drive — this is where the processed dataset, model checkpoints, and
# evaluation reports get backed up so they survive between Colab sessions.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Config: where the code lives locally on this VM (WORK_DIR), where persistent
# storage lives on Drive (DRIVE_ROOT), and where to get the code from (REPO_URL).
import os

PROJECT_NAME = "afrimeet-ai"
WORK_DIR = f"/content/{PROJECT_NAME}"
DRIVE_ROOT = "/content/drive/MyDrive/AfriMeet_AI"

# Public repo -> plain clone, no auth needed. Clear this (set to "") to fall back to
# the Drive-zip method instead.
REPO_URL = "https://github.com/claverfred/afrimeet-ai.git"
CODE_ZIP_PATH = f"{DRIVE_ROOT}/afrimeet-ai-code.zip"

os.makedirs(DRIVE_ROOT, exist_ok=True)
print("WORK_DIR:", WORK_DIR)
print("DRIVE_ROOT:", DRIVE_ROOT)

In [ ]:
# Get the project code onto this VM: clone from GitHub (default), or unpack a
# Drive-uploaded zip if REPO_URL was cleared above.
import shutil
import subprocess

if REPO_URL:
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    subprocess.run(["git", "clone", REPO_URL, WORK_DIR], check=True)
else:
    assert os.path.exists(CODE_ZIP_PATH), (
        f"{CODE_ZIP_PATH} not found. Run `python scripts/package_for_colab.py` "
        "locally and upload the resulting zip to that Drive path, or set REPO_URL above."
    )
    os.makedirs(WORK_DIR, exist_ok=True)
    shutil.unpack_archive(CODE_ZIP_PATH, WORK_DIR)
    print(f"Unpacked {CODE_ZIP_PATH} -> {WORK_DIR}")

In [ ]:
# cd into the cloned/unpacked repo and install the heavy ML + API dependencies
# (Colab's default environment doesn't have these — this is separate from your
# local .venv).
%cd $WORK_DIR
!pip install -q -r requirements/ml.txt -r requirements/api.txt
!pip install -q -e . --no-deps

# The editable install above writes a .pth file that only gets picked up by a *new*
# Python process (which is why the `!python scripts/...` calls later in this notebook
# work fine) — this kernel is already running, so `import afrimeet` won't see it until
# we add src/ to sys.path directly.
import sys
sys.path.insert(0, f"{WORK_DIR}/src")

In [ ]:
# Sanity check: confirm PyTorch sees the GPU and the afrimeet package imports
# correctly before running any real work below.
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

import afrimeet
print("afrimeet package loaded, version", afrimeet.__version__)

## Phase 2 — Data

Restores the processed dataset from a Drive-cached zip if one exists (fast, no
re-download). Otherwise downloads + prepares it fresh on the local Colab disk (fast
I/O — writing thousands of small audio files directly to a Drive-mounted path is
much slower) and caches the result back to Drive for next time.

In [ ]:
# Phase 2 — Data: restore the processed dataset from a Drive-cached zip if one
# exists; otherwise download + prepare it fresh (calls scripts/download_data.py and
# scripts/prepare_dataset.py, the exact same scripts used locally) and cache the
# result to Drive for next time.
import shutil
from pathlib import Path

data_zip = Path(DRIVE_ROOT) / "data_processed.zip"
processed_dir = Path(WORK_DIR) / "data" / "processed"

if data_zip.exists():
    print(f"Restoring processed dataset from {data_zip} ...")
    processed_dir.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(data_zip), str(processed_dir))
else:
    print("No cached dataset on Drive — downloading and preparing from scratch.")
    !python scripts/download_data.py
    !python scripts/prepare_dataset.py
    print(f"Caching processed dataset to {data_zip} for future sessions ...")
    shutil.make_archive(str(data_zip.with_suffix("")), "zip", root_dir=str(processed_dir))

## Phase 3 — Baseline evaluation (pre-trained Whisper)

In [ ]:
# Locate a processed test-split manifest to evaluate both models against
# (used by both the Phase 3 baseline eval and the Phase 5 fine-tuned eval below).
import glob

test_manifests = glob.glob(f"{WORK_DIR}/data/processed/*/test/manifest.csv")
assert test_manifests, "No test manifest found under data/processed/*/test/manifest.csv"
TEST_MANIFEST = test_manifests[0]
print("Using test manifest:", TEST_MANIFEST)

In [ ]:
# Run the pre-trained ("baseline") Whisper model through evaluate.py -> writes
# reports/metrics/baseline_summary.json and baseline_per_example.csv (WER/CER/latency).
!python scripts/evaluate.py --manifest "$TEST_MANIFEST" --model openai/whisper-small --run-name baseline

## Phase 4 — Fine-tune Whisper on the conference-domain data

Hyperparameters come from `configs/config.yaml` (`training:` section). Lower
`train_batch_size` there if you hit a CUDA out-of-memory error on the T4's 16GB.

If a fine-tuned model backup already exists on Drive from a previous session, this
restores it instead of retraining — set `RETRAIN = True` to force a fresh run.

In [ ]:
# Phase 4 setup: if a fine-tuned model was already backed up to Drive in a previous
# session, restore it and skip retraining. Set RETRAIN = True to force a fresh run
# (e.g. after changing hyperparameters in configs/config.yaml).
import shutil
from pathlib import Path

finetuned_dir = Path(WORK_DIR) / "models" / "finetuned"
backup_zip = Path(DRIVE_ROOT) / "models_finetuned.zip"

RETRAIN = False

if backup_zip.exists() and not RETRAIN:
    print(f"Found existing backup at {backup_zip} — restoring instead of retraining.")
    finetuned_dir.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(backup_zip), str(finetuned_dir))
    SKIP_TRAINING = True
else:
    SKIP_TRAINING = False

In [ ]:
# Fine-tune Whisper via train.py (uses configs/config.yaml's training: hyperparameters),
# unless a restored model made this unnecessary (see cell above).
if not SKIP_TRAINING:
    !python scripts/train.py
else:
    print("Skipping training — using the model restored from Drive. Set RETRAIN = True above to force retraining.")

In [ ]:
# Back up the fine-tuned model to Drive so it survives past this session (safe to
# re-run any time, including mid-training from a second cell if you're worried
# about a disconnect on a long run).
backup_path = shutil.make_archive(str(backup_zip.with_suffix("")), "zip", root_dir=str(finetuned_dir))
print(f"Backed up fine-tuned model to {backup_path}")

## Phase 5 — Evaluate the fine-tuned model and compare against baseline

In [ ]:
# Resolve the fine-tuned model's path from configs/config.yaml (same config the
# training cell used to decide where to save it).
from afrimeet.utils.config import load_config

config = load_config()
FINETUNED_MODEL = f"{config['paths']['models_finetuned']}/{config['whisper']['finetuned_model_name']}"
print("Using fine-tuned model:", FINETUNED_MODEL)

In [ ]:
# Run the fine-tuned model through the same evaluate.py used for the baseline ->
# writes reports/metrics/finetuned_summary.json and finetuned_per_example.csv.
!python scripts/evaluate.py --manifest "$TEST_MANIFEST" --model "$FINETUNED_MODEL" --run-name finetuned

In [ ]:
# Compare the baseline vs. fine-tuned summaries side by side (WER/CER relative
# improvement) -> writes reports/metrics/comparison.csv.
!python scripts/compare_models.py --runs baseline=reports/metrics/baseline_summary.json finetuned=reports/metrics/finetuned_summary.json

In [ ]:
# Back up all evaluation reports (baseline + fine-tuned summaries, comparison table)
# to Drive so they're not lost when this session ends.
import shutil
from pathlib import Path

reports_src = Path(WORK_DIR) / "reports" / "metrics"
reports_backup = Path(DRIVE_ROOT) / "reports_metrics"
if reports_backup.exists():
    shutil.rmtree(reports_backup)
shutil.copytree(reports_src, reports_backup)
print(f"Backed up metrics to {reports_backup}")

## Resuming in a later session

Just re-run the cells from the top:
- The data cell finds `data_processed.zip` on Drive and skips re-downloading.
- The training cell finds `models_finetuned.zip` on Drive and skips retraining
  (restores the model instead). Set `RETRAIN = True` to fine-tune again — e.g. after
  changing hyperparameters in `configs/config.yaml`.

For a long training run, Trainer also checkpoints locally to
`models/finetuned/.../checkpoint-N` every `save_steps` — but that's on the ephemeral
Colab disk, so it's lost on a disconnect mid-run. Re-run the "back up the fine-tuned
model to Drive" cell periodically during a long run if you want that protection.